In [ ]:
%config InlineBackend.figure_format = 'retina'
import os
import sys
import importlib
from pathlib import Path
root_path = Path(os.getcwd())
sys.path.append(str(Path('..').resolve()))

import math
import polars as pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from loader import load_player_data_for_scoring
from rosters_2026 import rosters_2026
from aggregation import aggregate_all
from steps.filter import filter_all, filtering_summary
from steps.decay import apply_decay, decay_summary
from steps.normalization import normalize_all, normalization_summary
from steps.segmentation import apply_segmentation, segmentation_summary
from steps.scoring import build_scores, scoring_summary

STATSBOMB_DIR = Path("..") / "data" / "Statsbomb"

In [67]:
# Build flat set of all rostered players
import rosters_2026
importlib.reload(rosters_2026)
from rosters_2026 import rosters_2026
rostered_players = {
    player 
    for squad in rosters_2026.values() 
    for player in squad.keys()
}
print(f"Total rostered players: {len(rostered_players)}")

Total rostered players: 374


In [ ]:
raw = load_player_data_for_scoring("recent_club_players")
clean = aggregate_all(raw, verbose=False)

In [ ]:
filtered = filter_all(clean, verbose=True)
filtering_summary(filtered)

In [ ]:
decayed = apply_decay(filtered, verbose=True)
decay_summary(decayed)

In [ ]:
for filename, df in decayed.items():
    metrics_in_file = [
        (k, v["column"]) 
        for k, v in __import__('player_score.player_metrics_config', fromlist=['PLAYER_METRICS']).PLAYER_METRICS.items()
        if v["file"] == filename and v["column"] in df.columns
    ]
    for metric_key, col in metrics_in_file:
        stats = df[col].drop_nulls()
        print(f"{metric_key}: min={stats.min():.3f}, median={stats.median():.3f}, max={stats.max():.3f}, skew={stats.skew():.2f}")

In [ ]:
import steps.normalization
importlib.reload(steps.normalization)
from steps.normalization import *
normalized = normalize_all(decayed, verbose=True)
normalization_summary(normalized)

In [ ]:
from player_score.steps.shrinkage import apply_shrinkage, shrinkage_summary

LINEUPS_PATH = Path("..") / "data" / "Statsbomb" / "lineups.parquet"

shrunk = apply_shrinkage(normalized, lineups_path=LINEUPS_PATH, verbose=True)
shrinkage_summary(shrunk)

In [ ]:
df = shrunk["advanced__player__network_centrality.csv"]
print(df.filter(pl.col("position_archetype").is_in(["CM", "DM"]))
      .group_by("position_archetype")
      .agg(pl.len().alias("count"))
      .sort("position_archetype"))

In [ ]:
segmented = apply_segmentation(shrunk, verbose=True)
segmentation_summary(segmented)

In [ ]:
import steps.scoring
importlib.reload(steps.scoring)
from steps.scoring import * 

scored = build_scores(segmented, verbose=True)
scoring_summary(scored, top_n=20)

In [ ]:
# Check raw events for name matches
#events = pd.read_parquet("../data/Statsbomb/events.parquet")
#matches = pd.read_parquet("../data/Statsbomb/matches.parquet")
sb_players = set(events["player"].dropna().unique())

matched = rostered_players & sb_players
unmatched = rostered_players - sb_players

print(f"Matched: {len(matched)}")
print(f"Unmatched: {len(unmatched)}")
print("\nSample unmatched:", list(unmatched)[:20])

Matched: 204
Unmatched: 170

Sample unmatched: ['Nahuel Molina', 'Dayot Upamecano', 'Guilherme Arana', 'Raúl Jiménez', 'Kylian Mbappé', 'Rúben Dias', 'Édouard Mendy', 'Nico Paz', 'Guglielmo Vicario', 'Renan Lodi', 'Lamine Yamal', 'Willian Pacho', 'Wilfred Ndidi', 'David Ospina', 'Lennart Karl', 'Bernardo Silva', 'Lionel Messi', 'Diogo Jota', 'Diogo Costa', 'Sandro Tonali']


In [70]:
from rapidfuzz import process, fuzz

sb_players_list = list(sb_players)

def find_best_match(name, candidates, threshold=80):
    result = process.extractOne(
        name, candidates, 
        scorer=fuzz.token_sort_ratio,
        score_cutoff=threshold
    )
    return result[0] if result else None

# Build mapping: roster_name → statsbomb_name
name_mapping = {}
unresolved = []

for name in unmatched:
    match = find_best_match(name, sb_players_list)
    if match:
        name_mapping[name] = match
    else:
        unresolved.append(name)

print(f"Fuzzy matched: {len(name_mapping)}")
print(f"Still unresolved: {len(unresolved)}")
print("\nSample mappings (verify these):")
for k, v in list(name_mapping.items())[:26]:
    print(f"  '{k}' → '{v}'")

Fuzzy matched: 26
Still unresolved: 144

Sample mappings (verify these):
  'Bruno Fernandes' → 'Brandon Fernandes'
  'Josh Sargent' → 'Joshua Sargent'
  'David Raya' → 'David Raum'
  'Luciano Rodríguez' → 'Guido Rodríguez'
  'Nicolás González' → 'Nicolás Iván González'
  'Joe Scally' → 'Joseph Scally'
  'Jérémy Doku' → 'Jeremy Doku'
  'Martín Zubimendi' → 'Martín Zubimendi Ibáñez'
  'Kenan Yıldız' → 'Kenan Yildiz'
  'Phil Foden' → 'Philip Foden'
  'Moïse Bombito' → 'Moise Bombito'
  'Karim Adeyemi' → 'Karim-David Adeyemi'
  'Daniel Muñoz' → 'Daniel Muñoz Mejía'
  'Facundo Pellistri' → 'Facundo Pellistri Rebollo'
  'Emiliano Martínez' → 'Damián Emiliano Martínez'
  'Aaron Ramsdale' → 'Aaron Ramsey'
  'Hernán Galíndez' → 'Hernán Ismael Galíndez'
  'Stephen Eustáquio' → 'Stephen Antunes Eustáquio'
  'Stanley Nwabali' → 'Stanley Bobo Nwabali'
  'Gonzalo Montiel' → 'Gonzalo Ariel Montiel'
  'Valentín Carboni' → 'Valentin Carboni'
  'Ferdi Kadıoğlu' → 'Ferdi Erenay Kadıoğlu'
  'Lautaro Martí

In [63]:
# Lower threshold to catch more, flag low-confidence matches
name_mapping_with_scores = {}
for name in unmatched:
    result = process.extractOne(
        name, sb_players_list,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=60  # lower threshold
    )
    if result:
        name_mapping_with_scores[name] = (result[0], result[1])

# Print all with scores for review
for k, (v, score) in sorted(name_mapping_with_scores.items(), key=lambda x: x[1][1], reverse=True):
    print(f"  {score:5.1f}  '{k}' → '{v}'")

   96.6  'Mohammed Salisu' → 'Mohamed Salisu'
   93.8  'Valentín Carboni' → 'Valentin Carboni'
   92.9  'Enzo Fernández' → 'Enzo Fernandez'
   92.3  'Nico Williams' → 'Neco Williams'
   92.3  'Josh Sargent' → 'Joshua Sargent'
   92.3  'Moïse Bombito' → 'Moise Bombito'
   91.7  'Sergiño Dest' → 'Sergino Dest'
   90.9  'Phil Foden' → 'Philip Foden'
   90.9  'Ismaël Koné' → 'Ismael Koné'
   90.0  'Marc Guéhi' → 'Marc Guehi'
   87.5  'Bruno Fernandes' → 'Brandon Fernandes'
   87.0  'Joe Scally' → 'Joseph Scally'
   86.5  'Davinson Sánchez' → 'Davinson Sánchez Mina'
   86.5  'Nicolás González' → 'Nicolás Iván González'
   86.4  'Aurélien Tchouaméni' → 'Aurélien Djani Tchouaméni'
   85.7  'Victor Boniface' → 'Victor Okoh Boniface'
   85.7  'Stanley Nwabali' → 'Stanley Bobo Nwabali'
   84.6  'Aaron Ramsdale' → 'Aaron Ramsey'
   84.2  'Santiago Giménez' → 'Santiago Tomás Giménez'
   83.9  'Thomas Partey' → 'Thomas Teye Partey'
   83.7  'Alejandro Grimaldo' → 'Alejandro Grimaldo García'
   83.3